# ASISTENTE PARA BOAS V4

Recomienda la clasificacion de pagos comparando los textos de un reporte FBL3N (SAP)
contra las plantillas del Glosario BOAS del pais correspondiente.

**El programa solo recomienda.** La actualizacion del documento historico es manual.

---

## Como funciona el motor

Cada plantilla del glosario se compone de dos tipos de contenido:

| En la plantilla | Trato del programa |
|---|---|
| Numeros, letras, `/`, `-`, `:`, espacios | **Ancla**: obligatorio, se compara literal y en orden |
| `*`, `(NUMERO)`, `?` | **Variable**: cualquier contenido, de cualquier largo (o ninguno) |

Dos capas conviven:

1. **Anclas** — se verifican sobre el **texto crudo**. Es la unica capa que puede
   distinguir dos categorias que solo difieren en unos digitos internos.
2. **Similitud** — se calcula sobre el **texto normalizado**. Como la normalizacion
   borra los bloques de digitos, esta capa es ciega al digito discriminador y por eso
   **nunca puede llegar a recomendacion directa** (techo de 84 puntos).

Si al menos una plantilla ancla, solo compiten las que anclaron. La similitud queda
apagada y sirve unicamente para ordenar entre ellas.

## Regla para escribir plantillas

> Antes de poner un asterisco sobre un digito, preguntate: *este numero es distinto
> cuando cambio de categoria, pero siempre igual dentro de la misma categoria?*
> Si la respuesta es si, es un **ancla**: va escrito literal, nunca dentro de un comodin.
> Si cambia linea por linea aunque sea del mismo grupo, es **variable**: va como `*`.

El error mas costoso es meter el digito discriminador dentro de un asterisco. Cuando eso
pasa, el programa pierde la unica senal que le permite separar dos categorias parecidas.

Ver la *Guia de Plantillas del Glosario BOAS* para el metodo completo.

---

**Nota de uso:** correr las celdas en orden. Las celdas 1 y 4 piden subir archivos.

## Celda 1 — Cargar FBL3N

In [ ]:
# ===== CELDA 1: Cargar FBL3N =====
from google.colab import files
import pandas as pd
import io

print("Sube el archivo FBL3N (.xlsx)")
subida = files.upload()

nombre_fbl3n = list(subida.keys())[0]
fbl3n = pd.read_excel(io.BytesIO(subida[nombre_fbl3n]), dtype=str)

print(f"\nArchivo cargado: {nombre_fbl3n}")
print(f"Filas: {len(fbl3n)}")
print(f"\nColumnas detectadas ({len(fbl3n.columns)}):")
for i, col in enumerate(fbl3n.columns):
    print(f"  [{i}] {repr(col)}")

fbl3n.head(3)

## Celda 2 — Validar company codes y mapear pais

Todo se lee como texto (`dtype=str`) a proposito: si pandas interpreta `Company Code`
o `Text` como numero, se pierden los ceros a la izquierda, que son anclas.

In [ ]:
# ===== CELDA 2: Validar company codes y mapear pais =====
MAPA_PAIS = {
    "7100": "BRASIL",          "7250": "PANAMA",
    "7260": "COSTA RICA",      "7271": "REP. DOMINICANA",
    "7351": "VENEZUELA",       "7600": "PARAGUAY",
    "7510": "MEXICO",          "7511": "MEXICO",
    "7512": "MEXICO",          "7513": "MEXICO",
    "7530": "MEXICO",          "7650": "COLOMBIA",
    "7660": "CHILE",           "7670": "PERU",
    "7700": "URUGUAY",
}

def limpiar_codigo(v):
    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s

fbl3n["_codigo"] = fbl3n["Company Code"].map(limpiar_codigo)
fbl3n["pais"] = fbl3n["_codigo"].map(MAPA_PAIS)
fbl3n["_texto_crudo"] = fbl3n["Text"].fillna("").astype(str).str.strip()
fbl3n["_sin_texto"] = fbl3n["_texto_crudo"] == ""

resumen = (fbl3n.groupby(["_codigo", "pais"], dropna=False)
                .agg(filas=("_codigo", "size"),
                     sin_texto=("_sin_texto", "sum"))
                .reset_index())
resumen["con_texto"] = resumen["filas"] - resumen["sin_texto"]
print(resumen.to_string(index=False))

sin_mapeo = sorted(fbl3n.loc[fbl3n["pais"].isna(), "_codigo"].unique())
if sin_mapeo:
    n = fbl3n["pais"].isna().sum()
    print(f"\n[AVISO] Codigos sin mapeo: {sin_mapeo} -> {n} filas se marcaran IGNORADAS")
else:
    print("\nTodos los company codes tienen mapeo.")

print(f"\nTotal filas: {len(fbl3n)}")
print(f"Filas sin texto (se ignoran): {fbl3n['_sin_texto'].sum()}")
print(f"Filas a clasificar: {(~fbl3n['_sin_texto'] & fbl3n['pais'].notna()).sum()}")
print(f"Textos crudos unicos: {fbl3n.loc[~fbl3n['_sin_texto'], '_texto_crudo'].nunique()}")

## Celda 3 — Inspeccion de textos crudos

Diagnostico. La *firma estructural* colapsa cada corrida de digitos a `D` y cada corrida
de letras a `A`, dejando los separadores intactos: sirve para ver cuantos formatos
distintos hay realmente en el archivo.

In [ ]:
# ===== CELDA 3: Inspeccion de textos crudos =====
import re
from collections import Counter

textos = fbl3n.loc[~fbl3n["_sin_texto"] & fbl3n["pais"].notna(), "_texto_crudo"]
conteo = textos.value_counts()

print(f"Textos unicos: {len(conteo)} | Filas: {conteo.sum()}\n")
print("--- TOP 15 mas repetidos ---")
for t, n in conteo.head(15).items():
    print(f"{n:>4}x  {t}")

print("\n--- 15 aleatorios (semilla fija) ---")
for t in conteo.index.to_series().sample(15, random_state=42):
    print(f"      {t}")

def firma(t):
    return re.sub(r"\d+", "D", re.sub(r"[A-Za-z]+", "A", t))

print("\n--- FORMATOS ESTRUCTURALES (firma -> n de textos unicos) ---")
for f, n in Counter(firma(t) for t in conteo.index).most_common(15):
    print(f"{n:>4}  {f}")

def sublote(t):
    m = re.match(r"\s*(\d{5,})", t)
    return m.group(1)[3:5] if m else "(sin bloque inicial)"

print("\n--- DIGITOS 4-5 DEL PRIMER BLOQUE ---")
for s, n in Counter(sublote(t) for t in conteo.index).most_common():
    print(f"{n:>4} textos unicos  ->  '{s}'")

## Celda 4 — Cargar glosario BOAS

El match de hoja se hace por nombre normalizado (sin tildes, sin puntos, sin espacios),
para que "REP. DOMINICANA" encuentre la hoja aunque este escrita "REPUBLICA DOMINICANA".

In [ ]:
# ===== CELDA 4: Cargar glosario BOAS =====
import unicodedata

def norm_nombre(s):
    s = str(s).strip().upper()
    s = "".join(c for c in unicodedata.normalize("NFD", s)
                if unicodedata.category(c) != "Mn")
    return re.sub(r"[^A-Z]", "", s)

print("Sube el glosario BOAS (.xlsx)")
subida_g = files.upload()
nombre_glos = list(subida_g.keys())[0]
libro = pd.ExcelFile(io.BytesIO(subida_g[nombre_glos]))

print(f"\nArchivo: {nombre_glos}")
print(f"Hojas ({len(libro.sheet_names)}): {libro.sheet_names}\n")

glosarios = {}
for hoja in libro.sheet_names:
    df = libro.parse(hoja, dtype=str)
    df.columns = [str(c).strip() for c in df.columns]
    glosarios[norm_nombre(hoja)] = (hoja, df)

paises_fbl3n = sorted(fbl3n.loc[fbl3n["pais"].notna(), "pais"].unique())

print("--- COBERTURA PAIS -> HOJA ---")
for p in paises_fbl3n:
    k = norm_nombre(p)
    if k in glosarios:
        hoja, df = glosarios[k]
        faltan = [c for c in ["BOA Concept", "CONCEPT", "ACTION"] if c not in df.columns]
        estado = f"OK, {len(df)} filas" if not faltan else f"[!] FALTAN columnas {faltan}"
        print(f"  {p:<18} -> hoja '{hoja}' | {estado}")
        print(f"       columnas reales: {list(df.columns)}")
    else:
        print(f"  {p:<18} -> [AVISO] sin hoja -> {(fbl3n['pais']==p).sum()} filas a REVISION MANUAL")

for p in paises_fbl3n:
    k = norm_nombre(p)
    if k in glosarios and "BOA Concept" in glosarios[k][1].columns:
        df = glosarios[k][1]
        print(f"\n--- PLANTILLAS {p} ---")
        for _, r in df.iterrows():
            print(f"  [{r.get('CONCEPT')}]  {repr(r.get('BOA Concept'))}")

## Celda 5 — Compilador de anclas

`compilar()` parte la plantilla por los comodines (`*`, `(NUMERO)`, `?` son equivalentes),
escapa cada segmento literal y los une con un comodin no-codicioso. La **cantidad** de
asteriscos no importa.

Modo `prefijo`: anclado al inicio, contenido libre al final.

`limpiar()` neutraliza los espacios duros (`\xa0`) que viajan al copiar desde SAP y que
se ven iguales a un espacio normal pero impiden el match.

In [ ]:
# ===== CELDA 5: Compilador de anclas + diagnostico del glosario =====
COMODINES = re.compile(r"\*+|\(\s*NUMERO[^)]*\)|\?+")
MIN_ANCLA_CHARS = 6

def limpiar(s):
    s = str(s).replace("\xa0", " ").replace("\u2007", " ").replace("\u202f", " ")
    return re.sub(r"\s+", " ", s).strip()

def compilar(plantilla, modo="prefijo"):
    p = limpiar(plantilla)
    trozos = COMODINES.split(p)
    piezas, anclas = [], []
    for i, t in enumerate(trozos):
        if t:
            piezas.append(re.escape(t)); anclas.append(t)
        if i < len(trozos) - 1:
            piezas.append(".*?")
    cuerpo = "".join(piezas)
    patron = {"estricto": f"^{cuerpo}$", "prefijo": f"^{cuerpo}", "libre": cuerpo}[modo]
    return {
        "regex": re.compile(patron, re.IGNORECASE),
        "n_chars": sum(len(a) for a in anclas),
        "n_digitos": sum(len(re.findall(r"\d", a)) for a in anclas),
    }

# --- Diagnostico opcional: cuantos textos toca cada plantilla, por modo ---
PAIS_DIAG = "URUGUAY"
k_diag = norm_nombre(PAIS_DIAG)
if k_diag in glosarios:
    hoja_diag = glosarios[k_diag][1]
    textos_diag = [limpiar(t) for t in conteo.index]
    N = len(textos_diag)
    print(f"Textos unicos a probar: {N}\n")
    print(f"{'CONCEPT':<28} {'anc':>4} {'dig':>4} {'estric':>7} {'prefij':>7} {'libre':>7}  plantilla")
    print("-" * 120)
    for _, r in hoja_diag.iterrows():
        pl = r.get("BOA Concept")
        if pd.isna(pl):
            continue
        m = {}
        for modo in ("estricto", "prefijo", "libre"):
            c = compilar(pl, modo)
            m[modo] = sum(1 for t in textos_diag if c["regex"].search(t))
        c = compilar(pl)
        alerta = ""
        if c["n_chars"] < MIN_ANCLA_CHARS:
            alerta = " <-- ANCLA CORTA"
        elif m["prefijo"] > 0.6 * N:
            alerta = " <-- DEMASIADO GENERICA"
        print(f"{str(r.get('CONCEPT'))[:27]:<28} {c['n_chars']:>4} {c['n_digitos']:>4} "
              f"{m['estricto']:>7} {m['prefijo']:>7} {m['libre']:>7}  {limpiar(pl)[:45]}{alerta}")

## Celda 6 — Banco de pruebas de plantillas candidatas

**Celda opcional.** Sirve para probar una plantilla *antes* de escribirla en el Excel:
mide cuantas filas capturaria y si se solapa con otra.

El bloque de solapamientos es el control clave: si dos plantillas de CONCEPT distinto
tocan el mismo texto, hay ambiguedad y conviene verla antes de que el motor la resuelva.

Las correcciones definitivas van **en el archivo del glosario**, no aqui.

In [ ]:
# ===== CELDA 6 (OPCIONAL): Banco de pruebas de plantillas candidatas =====
pares = [(limpiar(t), int(n)) for t, n in conteo.items()]
TOTAL_FILAS = sum(n for _, n in pares)

# Editar libremente para probar plantillas nuevas antes de llevarlas al Excel
CANDIDATAS = {
    # "MI_CONCEPT": "mi*plantilla*de*prueba",
}

if CANDIDATAS:
    print(f"Base: {len(pares)} textos unicos / {TOTAL_FILAS} filas\n")
    resultados = {}
    for concepto, pl in CANDIDATAS.items():
        c = compilar(pl, "prefijo")
        hits = [(t, n) for t, n in pares if c["regex"].search(t)]
        resultados[concepto] = {t for t, _ in hits}
        filas = sum(n for _, n in hits)
        print(f"[{concepto}]  anclas={c['n_chars']}")
        print(f"  {pl}")
        print(f"  -> {len(hits)} textos unicos | {filas} filas ({filas/TOTAL_FILAS:.1%})")
        for t, n in hits[:3]:
            print(f"       {n}x {t}")
        print()

    claves = list(resultados)
    print("--- SOLAPAMIENTOS ENTRE CANDIDATAS ---")
    hay = False
    for i in range(len(claves)):
        for j in range(i + 1, len(claves)):
            inter = resultados[claves[i]] & resultados[claves[j]]
            if inter:
                hay = True
                print(f"  [!] {claves[i]} vs {claves[j]}: {len(inter)} textos en conflicto")
                print(f"      ej: {list(inter)[0]}")
    if not hay:
        print("  Ninguno. Las candidatas son mutuamente excluyentes.")

    cubiertos = set().union(*resultados.values()) if resultados else set()
    falt = sorted(((t, n) for t, n in pares if t not in cubiertos), key=lambda x: -x[1])
    print(f"\n--- SIN CUBRIR: {len(falt)} textos / {sum(n for _, n in falt)} filas ---")
    for t, n in falt[:20]:
        print(f"  {n:>3}x {t[:90]}")
else:
    print("Sin candidatas que probar. Agregar entradas a CANDIDATAS si se quiere usar.")

## Celda 7 — Normalizacion y puntaje de similitud

`normalizar()` manda los bloques de 6+ digitos a `#` y los de 2-5 a `@` — simbolos
distintos, para conservar la estructura sin conservar los digitos.

**Esta funcion no puede distinguir dos categorias que solo difieran en digitos internos,
y no debe intentarlo.** Para eso estan las anclas, que corren sobre el texto crudo.

Incluye la **regla de contencion**: si una cadena esta contenida entera en la otra, el
puntaje arranca en 60 y sube segun cobertura.

In [ ]:
# ===== CELDA 7: Normalizacion y puntaje de similitud =====
from difflib import SequenceMatcher

def normalizar(t):
    """Solo para el puntaje de similitud. NUNCA para agrupar ni para anclas."""
    s = limpiar(t).upper()
    s = re.sub(r"\d{6,}", "#", s)   # bloques largos -> comodin
    s = re.sub(r"\d{2,5}", "@", s)  # bloques cortos -> otro comodin
    return s

def jaccard(a, b):
    ta = set(re.findall(r"[A-Z@#]+", a))
    tb = set(re.findall(r"[A-Z@#]+", b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

def puntaje_similitud(texto, plantilla):
    a, b = normalizar(texto), normalizar(plantilla)
    if not a or not b:
        return 0.0
    if a in b or b in a:
        cobertura = min(len(a), len(b)) / max(len(a), len(b))
        return 60 + 40 * cobertura        # regla de contencion
    sec = SequenceMatcher(None, a, b).ratio()
    return 100 * (0.6 * sec + 0.4 * jaccard(a, b))

print("ok celda 7")

## Celda 8a — Preparar glosario

Compila las plantillas de una hoja una sola vez. Las que no alcanzan el minimo de
caracteres ancla se apartan, para evitar que plantillas casi vacias hagan match con todo.

In [ ]:
# ===== CELDA 8a: preparar glosario =====
MIN_ANCLA_CHARS = 6
PISO_ANCLA = 90
TECHO_SIMILITUD = 84

def preparar_glosario(df):
    items, descartadas = [], []
    for _, r in df.iterrows():
        pl = r.get("BOA Concept")
        if pd.isna(pl) or not limpiar(pl):
            continue
        c = compilar(pl, "prefijo")
        reg = {"plantilla": limpiar(pl), "concept": r.get("CONCEPT"),
               "action": r.get("ACTION"), **c}
        if c["n_chars"] < MIN_ANCLA_CHARS:
            descartadas.append(reg)
        else:
            items.append(reg)
    return items, descartadas

print("ok 8a")

## Celda 8b — Motor de decision

**Regla de precedencia.** Si al menos una plantilla ancla, solo compiten las que
anclaron: la similitud queda apagada y sirve unicamente para ordenar entre ellas. Esto
es lo que impide que una plantilla de otra categoria gane por parecido.

`TECHO_SIMILITUD = 84` es la segunda proteccion: sin anclas nunca se cruza el umbral de
85, asi que **la similitud sola jamas recomienda directo**. Su techo es "revisar
candidatos".

Si dos plantillas anclan con la misma especificidad pero distinto CONCEPT, hay ambiguedad
real y la fila va a "revisar candidatos" aunque haya match.

In [ ]:
# ===== CELDA 8b: motor de decision =====
def clasificar(texto, items):
    t = limpiar(texto)
    anclan = [g for g in items if g["regex"].search(t)]
    empate = False

    if anclan:
        anclan.sort(key=lambda g: (-g["n_chars"], -g["n_digitos"]))
        cands = [(max(PISO_ANCLA, puntaje_similitud(t, g["plantilla"])), g) for g in anclan]
        if len(anclan) > 1:
            empate = (anclan[0]["n_chars"] == anclan[1]["n_chars"]
                      and anclan[0]["concept"] != anclan[1]["concept"])
        via = "ancla"
    else:
        todos = [(min(TECHO_SIMILITUD, puntaje_similitud(t, g["plantilla"])), g) for g in items]
        cands = sorted(todos, key=lambda x: -x[0])[:3]
        via = "similitud"

    if not cands:
        return {"banda": "revision manual", "puntaje": 0, "concept": None,
                "action": None, "alt_2": None, "alt_3": None, "via": "sin glosario"}

    pt, mejor = cands[0]
    if empate:
        banda = "revisar candidatos"
    elif pt >= 85:
        banda = "recomendacion directa"
    elif pt >= 60:
        banda = "revisar candidatos"
    else:
        banda = "revision manual"

    directa = (banda == "recomendacion directa")
    alts = [f"{g['concept']} ({p:.0f})" for p, g in cands[1:3]]
    return {
        "banda": banda,
        "puntaje": round(pt, 1),
        "concept": mejor["concept"] if banda != "revision manual" else None,
        "action": mejor["action"] if directa else None,
        "alt_2": alts[0] if (not directa and len(alts) > 0) else None,
        "alt_3": alts[1] if (not directa and len(alts) > 1) else None,
        "via": via,
    }

print("ok 8b")

## Celda 9 — Auditoria de glosario

Control de calidad del glosario. Aplica a cualquier pais: se le pasa la hoja y los textos
de ese pais.

Reporta plantillas sin match, plantillas demasiado genericas, conflictos, y que volumen
queda descubierto ordenado por filas — esa ultima lista es la cola de trabajo priorizada
para escribir plantillas nuevas.

**Que una plantilla no matchee no significa que este mal escrita.** Puede ser que esa
categoria simplemente no aparecio en este archivo. Antes de corregir nada, verificar si la
categoria existe en los datos.

In [ ]:
# ===== CELDA 9: Auditoria de glosario (aplica a cualquier pais) =====
def auditar(items, textos_unicos, conteos, nombre_pais):
    N = len(textos_unicos)
    TOT = sum(conteos)
    print(f"=== AUDITORIA {nombre_pais}: {len(items)} plantillas | {N} textos | {TOT} filas ===\n")

    cobertura, sin_match, genericas = {}, [], []
    for g in items:
        idx = [i for i, t in enumerate(textos_unicos) if g["regex"].search(t)]
        filas = sum(conteos[i] for i in idx)
        cobertura[id(g)] = set(idx)
        if not idx:
            sin_match.append(g)
        elif len(idx) > 0.6 * N:
            genericas.append((g, len(idx), filas))

    if sin_match:
        print(f"[?] {len(sin_match)} PLANTILLAS SIN MATCH EN ESTE ARCHIVO")
        print("    Puede ser error de escritura O que la categoria no salio esta vez.")
        print("    Verificar en los datos antes de corregir.\n")
        for g in sin_match:
            print(f"     [{g['concept']}] {g['plantilla'][:70]}")
    if genericas:
        print(f"\n[!] {len(genericas)} PLANTILLAS DEMASIADO GENERICAS (>60% de los textos):")
        for g, n, f in genericas:
            print(f"     [{g['concept']}] {g['plantilla'][:60]} -> {n} textos")

    print("\n[!] CONFLICTOS (mismo texto, CONCEPT distinto, misma especificidad):")
    hay = False
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            a, b = items[i], items[j]
            if a["concept"] == b["concept"]:
                continue
            inter = cobertura[id(a)] & cobertura[id(b)]
            if inter and a["n_chars"] == b["n_chars"]:
                hay = True
                print(f"     [{a['concept']}] vs [{b['concept']}]: {len(inter)} textos")
    if not hay:
        print("     Ninguno.")

    cub = set().union(*cobertura.values()) if cobertura else set()
    sin = [i for i in range(N) if i not in cub]
    filas_sin = sum(conteos[i] for i in sin)
    print(f"\nCOBERTURA POR ANCLAS: {TOT-filas_sin}/{TOT} filas ({(TOT-filas_sin)/TOT:.1%})")
    print(f"SIN CUBRIR: {len(sin)} textos / {filas_sin} filas - top 10 por volumen:")
    for i in sorted(sin, key=lambda i: -conteos[i])[:10]:
        print(f"   {conteos[i]:>3}x {textos_unicos[i][:80]}")

# Auditar cada pais presente en el FBL3N
txts = [t for t, _ in pares]
cnts = [n for _, n in pares]
for p in paises_fbl3n:
    k = norm_nombre(p)
    if k in glosarios and "BOA Concept" in glosarios[k][1].columns:
        it, _ = preparar_glosario(glosarios[k][1])
        sub_p = fbl3n[(fbl3n["pais"] == p) & (~fbl3n["_sin_texto"])]
        vc = sub_p["_texto_crudo"].map(limpiar).value_counts()
        auditar(it, list(vc.index), list(vc.values), p)
        print("\n" + "=" * 70 + "\n")

## Celda 10 — Pipeline multi-pais

Cada fila se enruta al glosario de **su propio** pais: el bucle solo carga la hoja
correspondiente, nunca compara contra las hojas de los demas paises.

El `cache` esta indexado por **texto crudo**. Ese detalle es la correccion estructural de
la falla de la version anterior: dos textos que normalizan igual pero difieren en el
crudo son entradas distintas y pueden recibir clasificaciones distintas.

In [ ]:
# ===== CELDA 10: Pipeline multi-pais =====
filas_ok = fbl3n[~fbl3n["_sin_texto"] & fbl3n["pais"].notna()].copy()
resultados_todos = []

for pais in sorted(filas_ok["pais"].unique()):
    sub = filas_ok[filas_ok["pais"] == pais]
    k = norm_nombre(pais)

    if k not in glosarios or "BOA Concept" not in glosarios[k][1].columns:
        print(f"[{pais}] sin hoja valida -> {len(sub)} filas a revision manual")
        for _, r in sub.iterrows():
            resultados_todos.append({**r.to_dict(), "banda": "revision manual",
                                     "puntaje": 0, "concept": None, "action": None,
                                     "alt_2": None, "alt_3": None, "via": "sin glosario"})
        continue

    items, desc = preparar_glosario(glosarios[k][1])
    cache = {}
    for _, r in sub.iterrows():
        t = r["_texto_crudo"]
        if t not in cache:                  # cache por TEXTO CRUDO
            cache[t] = clasificar(t, items)
        resultados_todos.append({**r.to_dict(), **cache[t]})

    print(f"[{pais}] {len(items)} plantillas activas ({len(desc)} descartadas) | "
          f"{len(sub)} filas | {len(cache)} textos unicos")

res = pd.DataFrame(resultados_todos)

print("\n=== METRICAS POR PAIS (% sobre filas) ===")
for pais, g in res.groupby("pais"):
    print(f"\n{pais}: {len(g)} filas | {g['_texto_crudo'].nunique()} textos unicos")
    for banda, n in g["banda"].value_counts().items():
        print(f"   {banda:<24} {n:>5} filas  {n/len(g):>6.1%}")
    print(f"   (via ancla: {(g['via']=='ancla').sum()} | via similitud: {(g['via']=='similitud').sum()})")

## Celda 11 — Tabla final y exportacion

Ordena por pais y luego por banda (recomendacion directa primero). Exporta con anchos
ajustados, paneles congelados y autofiltro.

El `print` de control cuadra el total contra las filas originales: si no coincide, alguna
fila se perdio en el camino.

In [ ]:
# ===== CELDA 11: Tabla final y exportacion =====
ignoradas = fbl3n[fbl3n["_sin_texto"] | fbl3n["pais"].isna()].copy()
for c in ["banda", "puntaje", "concept", "action", "alt_2", "alt_3", "via"]:
    ignoradas[c] = None
ignoradas["banda"] = "ignorada"
ignoradas["pais"] = ignoradas["pais"].fillna("SIN MAPEO")

final = pd.concat([res, ignoradas], ignore_index=True)
final = final.rename(columns={"concept": "CONCEPT", "action": "ACTION"})
final["ACCION REAL"] = ""

ORDEN_BANDA = {"recomendacion directa": 0, "revisar candidatos": 1,
               "revision manual": 2, "ignorada": 3}
final["_ord"] = final["banda"].map(ORDEN_BANDA)
final = final.sort_values(["pais", "_ord", "puntaje"], ascending=[True, True, False])

COLS = ["pais", "Account", "Document Number", "Text", "banda", "puntaje",
        "CONCEPT", "ACTION", "alt_2", "alt_3", "ACCION REAL"]
final = final[COLS]

salida = "RECOMENDACION_BOAS.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as w:
    final.to_excel(w, index=False, sheet_name="RECOMENDACIONES")
    ws = w.sheets["RECOMENDACIONES"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    anchos = {"pais": 16, "Account": 12, "Document Number": 16, "Text": 55,
              "banda": 22, "puntaje": 9, "CONCEPT": 26, "ACTION": 40,
              "alt_2": 24, "alt_3": 24, "ACCION REAL": 26}
    for i, c in enumerate(COLS, start=1):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = anchos[c]

print(f"Filas exportadas: {len(final)} (total original: {len(fbl3n)})")
print(final["banda"].value_counts().to_string())
files.download(salida)